# Software engineering agents: loops, evaluation, and interfaces

> Earlier lectures each treated one family of Agent tasks: lecture 07 let an Agent evolve a policy through self-improvement, lecture 08 turned search and deep research into an Agent that calls a browser, and lecture 09 finished post-training. Those methods were first verified on synthetic tasks: math problems, retrieval QA, maze navigation.
>
> This lecture moves the same Agent loop onto real software-engineering tasks: have the Agent fix a bug in a real repository, or write a high-performance GPU kernel. A real codebase can run to a million tokens, and feedback also comes from a real system: compile errors, test failures, and performance profiles are ready-made signals. We proceed along three nested problems: how much accuracy test-time compute can buy on code tasks; how evaluation should be designed when "correct" is not enough and "fast" is also required; and what interface aligns the Agent with the underlying system.

**SWE-bench** collects issues from real GitHub repositories, paired with tests that can automatically judge right or wrong. The Agent outputs a **patch**, and we run the tests to see whether they pass. This lecture's lead system, CodeMonkeys, cuts "fix one issue" into three stages: find context, generate candidates, select an answer.

Look at a concrete number for selection loss. On a real GitHub issue, the model first generates a large batch of candidate patches and puts them all into a candidate pool.

If we pick one at random from the pool and submit it, accuracy is only 45.8%.

If we switch to an oracle that "always picks correctly", accuracy jumps to 69.8%.

The gap between 45.8% and 69.8% is what the "select" step throws away: the correct answer is already in the pool, yet the system cannot pick it out.

The counterpart of selection is generation. On the same problem, if we let the model generate a few more patches, the probability that at least one of them is a correct fix grows approximately log-linearly with the number of samples. That regularity appeared on the math problems of lecture 2. Code tasks keep it, but add a layer of complexity: fixing an issue is not outputting one answer; it is producing a code change that passes the official tests.

Section 1 of this lecture starts from the generation stage, and looks at how much coverage **test-time compute** can buy on code tasks.

## 1. Test-time compute on code tasks

This section answers two things: if we generate a few more candidate patches, how much the fraction of correct fixes can rise; and given a budget, how to split it between serial and parallel.

Lecture 2 used the same idea on math problems: repeated sampling can push the fraction of correct fixes toward 1, growing approximately log-linearly with the number of samples. Code tasks keep that regularity, but add a layer of complexity: fixing an issue is not outputting one answer; it is producing a code change that passes the official tests. CodeMonkeys splits "fix one issue" into three subtasks that can be scaled separately. This section first treats the generation stage: how the fraction of correct fixes grows with the number of samples.

This section introduces a metric called coverage. Coverage is the fraction of a batch of problems for which at least one candidate patch is a correct fix. It measures the ceiling of the "generation stage": whether the candidate pool contains a correct answer at all.

We first compute a two-problem example by hand. Let the probability that a single candidate patch is a correct fix be p, and sample n independent patches. The probability that every patch is wrong is $(1-p)^n$, so the probability that at least one is a correct fix is $1-(1-p)^n$. For n = 2 and n = 5:

| Problem | Single-patch accuracy p | Pass rate at n=2 | Pass rate at n=5 |
|---|---|---|---|
| Problem 1 | 0.2 | $1-0.8^2=0.36$ | $1-0.8^5\approx 0.67$ |
| Problem 2 | 0.9 | $1-0.1^2=0.99$ | $1-0.1^5\approx 1.00$ |

Coverage is the average of the two problems' pass rates: at n=2, $(0.36+0.99)/2=0.675$; at n=5, about $0.835$. On the same batch of problems, increasing only the number of samples raises coverage from 0.675 to 0.835. Below we verify these two numbers in code, then generate a batch of problems of mixed difficulty and watch coverage grow with the number of samples.

In [ ]:
import numpy as np

# Pass rates of two problems: p1=0.2, p2=0.9, n independent samples, coverage is the mean of the two problems
p = np.array([0.2, 0.9])
for n in (2, 5):
    per_issue = 1 - (1 - p) ** n
    print("n=%d  per-problem pass rate %s  coverage %.3f" % (n, per_issue, per_issue.mean()))
print("Hand calculation agrees: n=2 coverage 0.675, n=5 coverage about 0.836.")


In [ ]:
# A batch of problems of mixed difficulty: single-patch accuracy p_i follows a right-skewed Beta
rng = np.random.default_rng(42)
n_problems = 200
p_i = rng.beta(1.5, 5.0, size=n_problems)
print("p_i mean %.3f, median %.3f: most problems are not solved by a single patch" % (p_i.mean(), np.median(p_i)))

samples = np.arange(1, 51)
coverage = np.array([(1 - (1 - p_i) ** k).mean() for k in samples])
print("k=1 coverage %.3f, k=50 coverage %.3f" % (coverage[0], coverage[-1]))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(samples, coverage)
ax.set_xscale("log")
ax.set_xlabel("sampled patches k")
ax.set_ylabel("coverage")
ax.set_title("Coverage grows with parallel sampling")
plt.show()
print("Key observation: coverage rises with the number of samples; the first few samples bring the largest gain, then the growth slows.")


The experiment above used only parallel sampling: each candidate patch is generated independently, with no interaction. A real system also uses serial repair: spend compute repeatedly on the same trajectory, generate a patch, run tests, take feedback, generate again, for several rounds. Trajectory is a term here for one attempt from start to finish. A real system uses both, so the problem becomes: given a budget, how to split it between serial and parallel.

We look at this allocation with a simplified model. A total budget of B calls is split into N trajectories, each doing r rounds of repair. The probability that a single trajectory is correct on the first generation is p; after a wrong generation, each repair round fixes it with probability $\eta$. We first compute this model by hand, then use code to watch coverage under different splits of the same budget.

First we look at what one call corresponds to. One call is one generation. Repair is an extra action after generation: given a total budget of B calls, split into N trajectories with r rounds of repair each, each trajectory actually produces r+1 calls — call 0 is the first generation, then r repair rounds, one each. That is the origin of the constraint $N(r+1)=B$.

The solve probability of a single trajectory is computed in two steps. The first generation is directly correct with probability p; only a wrong generation (probability $1-p$) enters repair. Each repair round independently fixes it with probability $\eta$, and the probability of remaining wrong is $1-\eta$. The probability that all r rounds remain wrong is $(1-\eta)^r$, so

$$\text{serial\_success} = 1-(1-p)(1-\eta)^r.$$

We compute one set of numbers by hand. Take p=0.2, $\eta=0.3$:

| Repair rounds r | Solve probability of a single trajectory |
|---|---|
| 0 | $1-0.8=0.20$ |
| 1 | $1-0.8\times0.7=0.44$ |
| 4 | $1-0.8\times0.7^4\approx0.81$ |

r=0 is pure parallel sampling, with no repair, and the solve probability equals the first-generation probability p. As r goes from 0 to 4, the same call budget is spent on one trajectory, and the solve probability rises from 0.20 to about 0.81. Repair makes the same budget produce more. The cost is fewer trajectories: at B=10, r=4 can open only $10/5=2$ parallel trajectories. The essence of budget allocation is a trade-off between "each trajectory is smarter" and "more trajectories".

In [ ]:
def serial_success(p, rounds, eta):
    """Solve probability of a single trajectory after rounds of serial repair.

    p: probability that a single generation is correct; eta: probability that each repair round fixes it.
    First generation fails with (1-p); each later round continues to fail with (1-eta).
    """
    return 1 - (1 - p) * (1 - eta) ** rounds

def split_coverage(p_i, n_traj, rounds, eta):
    """Coverage after n_traj parallel trajectories, each with rounds of repair."""
    s = serial_success(p_i, rounds, eta)
    return 1 - (1 - s) ** n_traj

# Three fixed splits: pure parallel / shallow repair / deep repair, compare growth with total budget
splits = {"parallel (r=0)": 1, "shallow (r=1)": 2, "deep (r=4)": 5}
budgets = np.arange(1, 41)
fig, ax = plt.subplots(figsize=(7, 4))
for label, calls in splits.items():
    covs = [split_coverage(p_i, max(1, B // calls), calls - 1, 0.3).mean()
            for B in budgets]
    ax.plot(budgets, covs, label=label)
ax.axhline(p_i.mean(), color="gray", ls="--", label="single generation")
ax.set_xlabel("total calls B")
ax.set_ylabel("coverage")
ax.set_title("Serial vs parallel budget allocation")
ax.legend(fontsize=8)
plt.show()

for label, calls in splits.items():
    cov = split_coverage(p_i, max(1, 10 // calls), calls - 1, 0.3).mean()
    print("%-16s B=10 coverage %.3f" % (label, cov))
print("Key observation: when the budget is tight, a split with repair is more cost-effective; once the budget is ample, the splits converge; "
      "single generation stays at a low level.")


Coverage is the ceiling of the generation stage. Even if the selection stage has an oracle — a hypothetical selector that always picks correctly — the final score cannot exceed coverage, because when the candidate pool has no correct answer, nobody has anything to pick. CodeMonkeys' three numbers tell this chain clearly: candidate-pool coverage is 69.8%, random selection is only 45.8%, and oracle selection equals coverage exactly. The selection method decides how much is recovered from that ceiling. Below we implement several selectors on a synthetic candidate pool and quantify each one's recovery fraction.

The synthetic data's protocol: 300 problems, 8 candidate patches each. Each candidate has a hidden correctness label, and has also been checked by 5 "generated tests": correct candidates have a higher per-test pass rate, and incorrect candidates occasionally slip through. That matches a real system: generated tests are not perfect, and that imperfection is what makes selection hard.

Walking coverage and selection through one concrete problem is clearer than looking at formulas alone. The problem: make compute_total(n) (return the sum from 0 to n) both correct and fast. The generation stage produced 4 candidate patches:

| Candidate | Implementation idea | Correctness | Speed |
|---|---|---|---|
| A | Arithmetic-series sum $n(n+1)/2$ | correct | O(1), fast |
| B | Sum formula missing a term $n(n-1)/2$ | wrong | O(1), fast |
| C | Always return 0 | fast but wrong | O(1), fastest |
| D | Accumulate one by one, sum(range(n+1)) | correct | O(n), slow |

The coverage stage asks only one thing: whether the pool contains a correct answer. A and D are both correct, so coverage holds for this pool. We look at how coverage filters a few pools. Pool 1 has {A, D}, pool 2 has only {B, C}, pool 3 has {A, B}. Pool 2 has no correct candidate; no matter how strong the selection stage is, this pool cannot be solved, and it is dropped from the coverage statistic. Coverage is "how many pools contain at least one correct candidate". In CodeMonkeys, oracle selection equals coverage exactly, for this reason: the oracle has something to pick only when the pool has a correct answer.

The selection stage picks one from the pool using a weak signal. Suppose there are only two generated tests, both asserting correctness only, and they cannot see speed:

| Candidate | Test 1: compute_total(0)==0 | Test 2: compute_total(100)==5050 | Passes |
|---|---|---|---|
| A | pass (0 is exactly 0) | pass (5050 exactly) | 2 |
| B | pass (the formula gives 0) | fail (gets 4950) | 1 |
| C | pass (always 0) | fail | 1 |
| D | pass | pass | 2 |

The vote is 2 each for A and D. The test signal cannot distinguish "correct and fast" from "correct but slow". That is what the next two sections address: fast_p continues to stratify correct candidates by speedup, and the interface design exposes the speed signal to the Agent explicitly. Under binary tests, a selector can only break the A/D tie with heuristics such as patch length or complexity. The argmax in the code below takes the first-appearing A.

Random selection picks uniformly among 4, with a hit rate of 1/2; test voting shrinks the answer to the two correct candidates; if the pool has only {B, C}, coverage is already zero. Measuring the two stages separately is what lets selection loss be quantified: the gap between CodeMonkeys' 45.8% and 69.8% is what the selection method dropped from the coverage ceiling.

In [ ]:
# Synthetic candidate pool: 300 problems × 8 candidates × 5 generated tests
M, C, T = 300, 8, 5
rng = np.random.default_rng(7)
q = rng.beta(1.2, 6.0, size=M)                # fraction of correct candidates per problem
correct = rng.random((M, C)) < q[:, None]      # hidden correctness labels
test_result = np.where(correct[:, :, None],
                       rng.random((M, C, T)) < 0.75,   # correct candidates pass tests at a high rate
                       rng.random((M, C, T)) < 0.20)   # incorrect candidates occasionally slip through

def random_score(correct):
    """Pick one candidate at random; fraction that hit a correct candidate."""
    pick = rng.integers(0, correct.shape[1], size=correct.shape[0])
    return correct[np.arange(correct.shape[0]), pick].mean()

def vote_score(correct, test_result):
    """Test majority vote: pick the candidate that passed the most generated tests."""
    votes = test_result.sum(axis=2)
    pick = np.argmax(votes, axis=1)
    return correct[np.arange(correct.shape[0]), pick].mean()

def oracle_score(correct):
    """Oracle selection: if a correct candidate exists it is always picked, i.e. coverage."""
    return correct.any(axis=1).mean()

cov = oracle_score(correct)
rand = random_score(correct)
vote = vote_score(correct, test_result)
print("random selection    %.3f" % rand)
print("test vote           %.3f" % vote)
print("oracle selection    %.3f (= coverage)" % cov)
print("vote recovers %.0f%% of the gap between random and the ceiling" % ((vote - rand) / (cov - rand) * 100))
print("Key observation: in the synthetic data the tests are nearly noiseless, so voting approaches the ceiling; "
      "in a real system tests are weaker and candidate outcomes are highly correlated, so the recovery fraction is much lower.")


## 2. Improving code with tests and performance feedback

The previous section's evaluation is binary: a patch that fixes the issue counts as correct. Many engineering tasks are not binary in success or failure: a function must be both correct and fast. This section treats that: how to measure "correct" and "fast" at the same time, and how to let an Agent make a function both fast and correct.

We use a benchmark called KernelBench. Its task gives the model a PyTorch reference implementation and requires the model to output a version with the same interface whose internals are replaced by a custom CUDA kernel, then automatically evaluates two axes: functional correctness (compare outputs on random inputs) and performance (speedup relative to PyTorch). CUDA is the programming language on a GPU; a kernel is a program that runs on the GPU; the reference implementation here is written in PyTorch, a common deep-learning framework. 250 tasks are split into three levels by operator count; Level 2 tests fusing a chain of operators into one kernel. This section first compresses "correct and fast" into a scalar metric fast_p, then looks at how test-time methods raise it.

fast_p is the fraction of a batch of candidate kernels that simultaneously satisfy "correct" and "speedup greater than a threshold p". Let $p$ be the speedup threshold. Among $N$ candidate kernels, that fraction is

$$\text{fast}_p = \frac{1}{N}\sum_{i=1}^{N} \mathbb{1}[\text{correct}_i \land \text{speedup}_i > p].$$

At $p=0$ it is pure accuracy; each time $p$ is raised a notch, another batch of "correct but not fast enough" candidates is filtered out. Hand calculation of four candidates:

| Candidate | Correct | Speedup | in fast_0 | in fast_1 | in fast_2 |
|---|---|---|---|---|---|
| A | yes | 3.2 | yes | yes | yes |
| B | yes | 0.8 | yes | no | no |
| C | no | 5.0 | no | no | no |
| D | yes | 1.5 | yes | yes | no |

fast_0 = 3/4, fast_1 = 2/4, fast_2 = 1/4. B is correct but slow: it enters fast_0 and cannot enter fast_1. C is fast but wrong: it cannot even enter fast_0. The two candidates each fail one axis, which is why "correct and fast" are both required.

Two symbols in the fast_p formula need to be read in order. $\mathbb{1}[\cdots]$ is the indicator function: 1 when the condition in the brackets holds, otherwise 0. $\land$ is logical and: both conditions must hold. The whole sentence reads: among all N candidates, count those that are both correct and more than p times faster than PyTorch, then divide by N.

Think of it as two gates, the first checking correctness, the second checking speedup. Candidate A (correct, 3.2×) passes both in order. B (correct, 0.8×) passes the first; the second requires faster than 1×, 0.8 is not enough, and it drops here. C (wrong, 5.0×) is very fast, but already fails the first gate. D (correct, 1.5×) passes both, but when p is raised to 2, 1.5 is again not enough. The threshold p sets the height of the second gate: at p=0 this gate almost always lets candidates through, and fast_0 degenerates to pure accuracy.

The condition is written $>p$, not $\ge p$. A candidate whose speedup equals p exactly is not counted. p=1 means "strictly faster than the baseline"; being exactly as fast as PyTorch does not count as meeting the bar. That is a boundary convention, and it affects only candidates that land exactly on the threshold, which can usually be ignored.

Compressing "correct" and "fast" into one scalar is so that evaluation yields a single number for ranking. Binary evaluation (right or wrong) cannot distinguish "correct but slow" from "correct and fast"; KernelBench must continue to stratify "correct" candidates by speedup. p is the evaluation knob: raise it and the task gets harder, lower it and it gets looser. The same batch of candidates yields a fast_p curve under different p, which makes it convenient to compare models and methods at different strictness.

In [ ]:
def fast_p(candidates, p):
    """Return the fraction of candidates that are correct and have speedup greater than p.

    candidates: a list of (is_correct, speedup) pairs.
    """
    correct = np.array([c[0] for c in candidates], dtype=bool)
    speedup = np.array([c[1] for c in candidates])
    return np.mean(correct & (speedup > p))

cands = [("A", True, 3.2), ("B", True, 0.8), ("C", False, 5.0), ("D", True, 1.5)]
pairs = [(c, s) for _, c, s in cands]
for p in (0.0, 1.0, 2.0):
    print("fast_%.1f = %.2f" % (p, fast_p(pairs, p)))
print("Key observation: raising p one notch filters out another batch of 'correct but slow' candidates.")


In [ ]:
# A synthetic batch of candidate kernels: correctness is random, speedup is log-normal
rng = np.random.default_rng(3)
n = 300
correct = rng.random(n) < 0.4
speedup = np.exp(rng.normal(0.2, 0.9, size=n))
pairs = list(zip(correct, speedup))

ps = np.linspace(0.0, 2.5, 101)
fasts = np.array([fast_p(pairs, p) for p in ps])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ps, fasts)
ax.axvline(1.0, color="gray", ls="--", label="p = 1")
ax.set_xlabel("speedup threshold p")
ax.set_ylabel("fast_p")
ax.set_title("fast_p shrinks as the threshold rises")
ax.legend()
plt.show()
print("fast_0=%.3f  fast_1=%.3f  fast_2=%.3f"
      % (fast_p(pairs, 0.0), fast_p(pairs, 1.0), fast_p(pairs, 2.0)))
print("Key observation: the threshold p is the evaluation knob; raising it makes the task harder, "
      "and this curve also makes it convenient to upgrade the baseline to torch.compile later.")


First we fix the target: have the model write a kernel that is both fast and correct, and look at how low the probability of succeeding in one try is. one-shot means letting the model generate only once. On KernelBench, frontier models in one-shot average under 20% of tasks faster than PyTorch Eager, PyTorch's default operator-by-operator execution. Writing a correct kernel is already hard; writing one that is correct and fast is harder.

Methods that spend more inference compute (that is, test-time compute) spend the budget on multiple calls. That is the approach of lecture 2. KernelBench compared two. One is repeated sampling: generate k candidates in parallel, and succeed if any one meets the bar. The other is iterative refinement: over several rounds, feed the previous round's generation G, execution feedback E, and profiler feedback P back to the model, so each round knows a little more than the last. At the same budget of 10 calls, iteration is stronger: DeepSeek-R1's Level 2 fast_1 (threshold p=1, i.e. strictly faster than PyTorch) rose from 36% one-shot to 72%, and the feedback combination G+E+P was the strongest. Below we use a simplified model to watch how feedback raises each round's success rate.

What the three letters refer to. G (generation) is the kernel code generated in the previous round. E (execution) is the previous round's run result, including whether it was correct and how many milliseconds it took. P (profiling) comes from a profiler, a tool that measures where a program spends time, reporting line by line the fraction of time each statement occupies. Python's built-in cProfile is one such tool; on GPU there are tools such as NVIDIA's ncu.

We look at how the three pieces of feedback relay on a concrete example. The task is to write a function that returns the maximum of an array.

- Round 1 generates implementation G₁: two nested loops, for each element scanning the whole array again to find the max, complexity O(n²). Execution feedback E₁: the result is correct, but at n=10000 it takes several seconds. Profiler feedback P₁: the inner scan on line 5 occupies 99% of the time.
- Round 2 the model reads P₁, recognizes that the inner scan is redundant, and generates G₂: one linear scan O(n). E₂: the result is correct, time drops to milliseconds.

Each round, the information fed back moves the next round a step closer to "correct and fast", and the per-round success rate climbs from p₀ toward p_max. Repeated sampling has no such mechanism: k independent samples each have the same success rate p, and the level does not rise with the number of calls. That is the source of iteration's strength at the same budget: DeepSeek-R1's Level 2 fast_1 rose from 36% one-shot to 72% because each round knew a little more than the last.

Below we quantify the climb with a simplified model: each round's success rate follows a saturation curve from p₀ toward p_max, and cumulative coverage takes the probability that "any one round succeeded". The richer the feedback combination (E+P), the higher p_max.

In [ ]:
def repeated_coverage(p, k):
    """Coverage of k independent samples."""
    return 1 - (1 - p) ** k

def iterative_cum(p0, p_max, rounds, tau=3):
    """Cumulative coverage of iterative refinement: round-t success rate climbs from p0 toward p_max.

    Per-round success probability p_t = p0 + (p_max - p0) * (1 - exp(-t/tau)),
    cumulative coverage is the probability that any one round succeeded.
    """
    t = np.arange(1, rounds + 1)
    p_t = p0 + (p_max - p0) * (1 - np.exp(-t / tau))
    return 1 - np.cumprod(1 - p_t)

B = 10
ks = np.arange(1, B + 1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, repeated_coverage(0.20, ks), label="repeated sampling")
ax.plot(ks, iterative_cum(0.20, 0.45, B), label="iterative (E)")
ax.plot(ks, iterative_cum(0.20, 0.60, B), label="iterative (E+P)")
ax.axhline(repeated_coverage(0.03, B), color="gray", ls=":", label="hard group")
ax.set_xlabel("calls")
ax.set_ylabel("fast_1 coverage")
ax.set_title("Execution and profiler feedback lift success")
ax.legend(fontsize=8)
plt.show()

print("Easy problems, 10 calls: repeated sampling %.3f, plus execution feedback %.3f, plus profiler %.3f"
      % (repeated_coverage(0.20, B), iterative_cum(0.20, 0.45, B)[-1],
         iterative_cum(0.20, 0.60, B)[-1]))
print("Hard problems, 10 calls: repeated sampling %.3f, plus execution feedback %.3f"
      % (repeated_coverage(0.03, B), iterative_cum(0.03, 0.05, B)[-1]))
print("Key observation: feedback raises each round's success rate, so iteration is stronger than repeated sampling; "
      "but for problems whose intrinsic success rate is too low, more feedback still does not help.")


## 3. Making the Agent's actions match the system interface

The first two sections put attention on the Agent's generation and selection, with an implicit assumption: the feedback the Agent reads is the system's true semantics. That assumption often does not hold. This section treats the case where the information the interface displays is inconsistent with the system's true semantics: how the Agent then errs, and how to align it.

We first give this phenomenon a name. The paper On the Need to Align Intent and Implementation calls it construct drift: uncertainty computed for A is used to support a conclusion about B. The engineering form is treating test pass rate as repair success, or high coverage as high accuracy. The Agent's intent must align with the interface's implementation semantics, or the metric will drift.

At the system layer, the paper Agent-System Interface gives two devices. A DSL (domain-specific language) makes "how to map" explicit as a searchable space. AutoGuide translates raw execution output into actionable natural-language advice. Below we use a minimal example to demonstrate the Agent's failure mode when the interface mismatches.

Interface here means the set of information the Agent can read after each action. The Agent can only decide from what it reads; information it cannot read does not exist for it. The interface decides the world as the Agent sees it. When the information the interface displays is inconsistent with the system's true semantics, the metric drifts. That is the construct drift mentioned above.

We first look at how behavior differs when the interface differs. Two Agents each bring an interface to fix the same bug. Interface 1 only returns "passed 1 of 3 tests"; the Agent can only know that 2 remain, with no way to judge where it went wrong. Interface 2 returns the failed assertion itself — "sum_evens([1, 2, 3, 4]) expected 6 got 4" — and the Agent sees the wrong value directly. The same system state, interface 2 gives strictly more information, so the next action is naturally different. Misalignment has a more hidden form as well: the interface says "pass", but the semantics are "did not crash" rather than "the result is correct". When tests only check that the function does not raise, and do not check the return value, any implementation that does not crash counts as passing. The model thinks it has fixed the bug, and the result is still wrong.

The interface paper gives two devices. The first is a DSL, a domain-specific language. Rather than letting the model improvise freely, we give it a fixed set of mapping primitives and compress feasible edits into a searchable space. KernelBench allowing Triton belongs to this class. Triton is a high-level programming language aimed at GPUs; its tiling and shared memory are handled implicitly by the compiler, so the model need not manage those details itself, and the kernels it writes are also more likely to compile. The second is AutoGuide, which handles the reverse information flow: translate raw profiler output — tens of thousands of lines of sampling reports — into actionable natural-language advice, such as "the loop on line 3 occupies 80% of the time; hoist the remainder operation out of the loop". What the model reads is ordinary language, not a raw sampling stack.

Below we use a minimal example to demonstrate interface misalignment: the same Agent, only the interface changes, and behavior goes from "pick the fastest" to "pick both fast and correct".

We design a minimal interface misalignment. The task: make compute_total(n) (return the sum from 0 to n) faster, and it must remain correct. The repository has three candidate implementations: a naive version (O(n)), a cheat version that is only fast and not correct (discard the input, always return 0), and a formula version that is both fast and correct (arithmetic-series sum). Two interfaces: interface A reports only a latency string, interface B reports a structured result — correctness, latency, and speedup relative to the baseline. The Agent's policy is to pick the "best" candidate according to the interface report. The interface decides which objective it treats as "good".

In [ ]:
import time

def compute_total_naive(n):
    return sum(range(n + 1))

def compute_total_fast_wrong(n):
    return 0                     # discard the input: latency near 0, but the result is wrong

def compute_total_fast_right(n):
    return n * (n + 1) // 2      # arithmetic-series sum: correct and O(1)

proposals = [("naive", compute_total_naive),
             ("fast_wrong", compute_total_fast_wrong),
             ("fast_right", compute_total_fast_right)]

def measure_ms(fn, n=20000, repeat=50, runs=5):
    """Time several runs and take the median; return the mean milliseconds of one call."""
    samples = []
    for _ in range(runs):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(n)
        samples.append((time.perf_counter() - t0) / repeat * 1000)
    return float(np.median(samples))

def harness_a(fn, n):
    """Interface A: return only a latency string, do not expose correctness."""
    return "%.3f ms" % measure_ms(fn, n)

def harness_b(fn, n, reference):
    """Interface B: structured report with correctness, latency, and speedup relative to the baseline."""
    ms = measure_ms(fn, n)
    baseline = measure_ms(reference, n)
    return {"correct": fn(n) == reference(n),
            "time_ms": ms,
            "speedup": baseline / ms if ms > 0 else float("inf")}

def agent_under_a(proposals, n):
    """Under interface A, pick a candidate by the latency string: take the minimum, ties take the first."""
    best_name, best_text = None, None
    for name, fn in proposals:
        text = harness_a(fn, n)
        if best_text is None or text < best_text:
            best_name, best_text = name, text
    return best_name, best_text

def agent_under_b(proposals, n, reference):
    """Under interface B, first filter for correctness, then pick by speedup."""
    best_name, best_speed = None, -1.0
    for name, fn in proposals:
        r = harness_b(fn, n, reference)
        if r["correct"] and r["speedup"] > best_speed:
            best_name, best_speed = name, r["speedup"]
    return best_name, best_speed

n = 20000
for name, fn in proposals:
    print("%-12s %.4f ms" % (name, measure_ms(fn, n)))

name_a, text_a = agent_under_a(proposals, n)
name_b, speed_b = agent_under_b(proposals, n, compute_total_naive)
correct_a = dict(proposals)[name_a](n) == n * (n + 1) // 2
correct_b = dict(proposals)[name_b](n) == n * (n + 1) // 2
print("Interface A picked %-11s external judgment correct = %s" % (name_a, correct_a))
print("Interface B picked %-11s external judgment correct = %s (speedup %.1fx)" % (name_b, correct_b, speed_b))
print("Key observation: interface A compresses 'fast' and 'correct' into one latency string, "
      "and the Agent has no correctness signal; interface B exposes correctness explicitly, "
      "so the Agent's intent aligns with the system semantics.")


## 4. The SWE-Agent loop: repair with two state machines

The first three sections are three pieces of a puzzle: test-time compute answers how to spend the budget, the evaluation metric answers how to set the target, and the interface answers how to align feedback. This section assembles the three pieces into a complete loop that can run on a toy repository. The goal is to turn the most central of "find context, generate candidates, select an answer" — generate candidates — into real code: read source, edit, run tests, take feedback, retry.

To make the most of the feedback, CodeMonkeys splits repair into two back-to-back state machines. A state machine here means two loops with different jobs. The first loop first generates a test that can reproduce the issue; the second loop takes that test as the judge, and lets editing and testing correct each other. Below we first build the toy repository, then the test state machine, then the edit state machine.

We first use a controllable toy repository, which makes the structure easier to see than going straight to a real repository. The repository has a module buggy.py, containing a defective function sum_evens(nums): it intends to add the even numbers in a list, but also accumulates the odd numbers into the total. Judging right or wrong can be programmatic: the correct implementation is fixed, and substituting the input is enough.

The test state machine's goal is to produce a test script that must fail on the unrepaired buggy.py, and must pass on the reference implementation. The first requirement is called reproducing the issue: a test failure means the bug was actually triggered. The second is called a two-sided check, ensuring the test can actually judge right or wrong. The edit state machine takes this test as the judge and iteratively modifies the source. The whole loop needs the LLM to supply two products: the test script and the modified source.

In [ ]:
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm
client = get_llm()

if False:
    print("LLM client ready, real API demo: tests and edits are provided by a scripted trajectory, "
          "and the loop skeleton is the same as a real API.")
else:
    print("LLM client ready, real API mode.")


In [ ]:
import tempfile

repo_dir = tempfile.mkdtemp(prefix="swe_repo_")
BUGGY = ('def sum_evens(nums):\n'
         '    """Return the sum of all even numbers in the list."""\n'
         '    total = 0\n'
         '    for x in nums:\n'
         '        if x % 2 == 1:      # Bug: odd numbers are also added into the total\n'
         '            total += x\n'
         '    return total\n')
FIXED = ('def sum_evens(nums):\n'
         '    """Return the sum of all even numbers in the list."""\n'
         '    total = 0\n'
         '    for x in nums:\n'
         '        if x % 2 == 0:      # Fix: add even numbers only\n'
         '            total += x\n'
         '    return total\n')

with open(os.path.join(repo_dir, "buggy.py"), "w") as f:
    f.write(BUGGY)

print("toy repo directory:", repo_dir)
print(BUGGY)


In [ ]:
import subprocess

def run_test(repo_dir, test_source, max_out=3):
    """Execute a test script in the repository directory and return (exit_code, output tail).

    repo_dir: repository directory; test_source: test-script source. The script
    imports a module in the repository to verify behavior. On assertion failure the exit code is nonzero.
    """
    test_path = os.path.join(repo_dir, "_run_test.py")
    with open(test_path, "w") as f:
        f.write(test_source)
    proc = subprocess.run(
        [sys.executable, "_run_test.py"], cwd=repo_dir,
        capture_output=True, text=True, timeout=30,
    )
    tail = (proc.stdout + proc.stderr).strip().splitlines()[-max_out:]
    return proc.returncode, tail

# Run a representative test on unrepaired buggy.py; it should reproduce the issue (nonzero exit)
code, tail = run_test(repo_dir, "from buggy import sum_evens\n"
                                 "assert sum_evens([1, 2, 3, 4]) == 6\nprint('PASS')")
print("exit=%d on unrepaired code" % code)
print("output tail:", tail)
print("Key observation: a nonzero exit code is the cheapest execution feedback.")


We walk the serial repair loop step by step on sum_evens. First read the code: sum_evens(nums) in buggy.py intends to accumulate even numbers, but the condition is written `x % 2 == 1`. `%` is remainder; `x % 2 == 1` holds when x is odd, so the function also adds odd numbers into total. On [1, 2, 3, 4] it computes 1+3=4; the correct answer is 2+4=6.

The first step of the loop is to generate a test. The test state machine produces a standalone script that asserts the correct behavior with assert:

- Round 1 proposes `assert sum_evens([]) == 0`. On an empty list any implementation returns 0, so this test passes on both the unrepaired code and the reference implementation. It cannot reproduce the bug, is rejected, and we try another round.
- Round 2 proposes `assert sum_evens([1, 2, 3, 4]) == 6`. On the unrepaired code it computes 4, the assertion fails (nonzero exit), and the issue is reproduced; on the reference implementation 6 equals 6, and it passes. The test is accepted.

The second step is mutual correction between editing and testing. The edit state machine reads the current source, modifies it, and runs this test:

- Round 1 of editing changes the condition to `x % 2 == 2`. For any integer x the remainder is only 0 or 1, so this condition is identically false and total is identically 0. Run the test: 0 is not equal to 6, nonzero exit, and the output tail carries the assertion-failure traceback.
- Feed "nonzero exit + output tail" back as feedback. Round 2 of editing changes the condition to `x % 2 == 0`. Run the test: get 6, exit 0, output PASS. The loop terminates and the source is accepted.

The whole process is read the code, edit, run tests, look at the output, edit again. Every step of feedback is a real execution result, and the model corrects the next step from it. Test and edit are split into two state machines, each with an independent acceptance criterion: the test requires a two-sided check (fail on unrepaired, pass on the reference implementation), and the edit requires the test to pass. The benefit of the split is that the test is first verified to be reliable, and the edit only answers a judge already known to be reliable, so errors of the two state machines do not contaminate each other.

In [ ]:
def extract_python(text):
    """Extract a ```python ... ``` code block from a reply; return an empty string if there is none."""
    start = text.find("```python")
    if start == -1:
        return ""
    start = text.find("\n", start) + 1
    end = text.find("```", start)
    return text[start:end] if end != -1 else text[start:]

def propose_test(issue, client, round_no):
    """Ask the LLM to generate a test script that can reproduce the issue.

    Under a real API demo this returns a scripted candidate: round 1 gives a weak test that
    both versions pass (it cannot catch the bug); round 2 gives a strong test that reproduces the bug.
    """
    if False:
        scripted = [
            "from buggy import sum_evens\n"
            "assert sum_evens([]) == 0\nprint('PASS')",
            "from buggy import sum_evens\n"
            "assert sum_evens([1, 2, 3, 4]) == 6\nprint('PASS')",
        ]
        return scripted[min(round_no, len(scripted) - 1)]
    prompt = ("The function sum_evens in the repository buggy.py is defective. Write a standalone "
              "Python test script that must fail its assertion on the unrepaired code.\n" + issue)
    reply = client.chat([{"role": "user", "content": prompt}])
    return extract_python(reply) or ""

def generate_test(issue, repo_dir, client, max_rounds=3):
    """Test state machine: iterate out a test script that can reproduce the issue.

    The acceptance standard is a two-sided check: the script must fail on the unrepaired code (reproduce the bug)
    and must pass on the reference implementation. Return (whether accepted, test script, per-round trace).
    """
    trace = []
    test = ""
    for r in range(max_rounds):
        test = propose_test(issue, client, r)
        code_buggy, _ = run_test(repo_dir, test)
        with open(os.path.join(repo_dir, "buggy.py"), "w") as f:
            f.write(FIXED)
        code_fixed, _ = run_test(repo_dir, test)
        with open(os.path.join(repo_dir, "buggy.py"), "w") as f:
            f.write(BUGGY)
        reproduced = code_buggy != 0
        verified = code_fixed == 0
        trace.append({"round": r, "reproduced": reproduced,
                      "verified": verified})
        if reproduced and verified:
            return True, test, trace
    return False, test, trace

print("Test state machine ready: extract_python / propose_test / generate_test")


In [ ]:
def propose_edit(current_source, feedback, client, round_no):
    """Ask the LLM to generate modified source given the current source and execution feedback.

    Under a real API demo this returns a scripted trajectory: round 1 gives a still-buggy edit
    (the predicate is changed to identically false); round 2 gives the correct fix.
    """
    if False:
        scripted = [
            current_source.replace("x % 2 == 1", "x % 2 == 2"),
            FIXED,
        ]
        return scripted[min(round_no, len(scripted) - 1)]
    prompt = ("Current source:\n" + current_source + "\nExecution feedback:\n" + feedback
              + "\nPlease output the full repaired buggy.py source, inside a ```python block.")
    reply = client.chat([{"role": "user", "content": prompt}])
    new_source = extract_python(reply)
    return new_source if new_source else current_source

def repair_loop(issue, repo_dir, test, client, max_rounds=4):
    """Edit state machine: read source, edit, run tests, take feedback, retry.

    The test script has already been produced and accepted by the test state machine. Each round
    feeds the exit code and output back to the model as feedback. Return (final source, whether fixed, per-round trace).
    """
    with open(os.path.join(repo_dir, "buggy.py")) as f:
        source = f.read()
    trace = []
    feedback = "Initial state; no test has been executed yet."
    for r in range(max_rounds):
        new_source = propose_edit(source, feedback, client, r)
        with open(os.path.join(repo_dir, "buggy.py"), "w") as f:
            f.write(new_source)
        code, tail = run_test(repo_dir, test)
        trace.append({"round": r, "exit": code, "tail": tail})
        if code == 0:
            return new_source, True, trace
        feedback = "exit=%d, output: %s" % (code, " | ".join(tail))
        source = new_source          # take the previous round's edit as the next round's starting point
    return new_source, False, trace

print("Edit state machine ready: propose_edit / repair_loop")


In [ ]:
issue = ("sum_evens(nums) should return the sum of all even numbers in the list, "
         "but the current implementation also counts odd numbers.")
ok_test, test, test_trace = generate_test(issue, repo_dir, client)
print("Test state machine: accepted = %s" % ok_test)
print("Test script:\n%s" % test)
for step in test_trace:
    print("  round %d  reproduced bug=%s  reference implementation passed=%s"
          % (step["round"], step["reproduced"], step["verified"]))

fixed, solved, edit_trace = repair_loop(issue, repo_dir, test, client)
print("Edit state machine: fixed = %s" % solved)
for step in edit_trace:
    print("  round %d  exit=%s  output=%s"
          % (step["round"], step["exit"], step["tail"]))


In [ ]:
# Independent verification with inputs the test state machine has not seen
oracle_test = ("from buggy import sum_evens\n"
               "assert sum_evens([]) == 0\n"
               "assert sum_evens([0, 2, 4, 6]) == 12\n"
               "assert sum_evens([1, 3, 5]) == 0\n"
               "assert sum_evens([1, 2, 3, 4, 5]) == 6\n"
               "print('ORACLE PASS')\n")
code, tail = run_test(repo_dir, oracle_test)
print("Independent oracle verification exit=%d, output=%s" % (code, tail))
print("Final source:\n" + fixed)
print("Key observation: the two state machines each handle half — the test is responsible for 'reproduce and judge', "
      "the edit for 'modify and submit'; under a real API demo the products come from a scripted trajectory, "
      "under a real API they are generated by the model, and the loop skeleton is unchanged.")


## Summary

This lecture moved the Agent from synthetic tasks onto real software-engineering tasks, and walked the scaffold of "find context, generate candidates, select an answer":

- [ ] Fixing an issue can be cut into three stages, each with an independent metric: context recall, generation coverage, selection score
- [ ] Coverage grows approximately log-linearly with the number of samples; the first few samples bring the largest gain
- [ ] Under the same total budget, different allocations of serial repair and parallel sampling yield similar coverage; when the budget is tight, repair is more cost-effective
- [ ] Coverage is the ceiling of generation; the selection method decides how much is recovered from that ceiling
- [ ] fast_p encodes correctness and speedup together with one threshold; p is the evaluation knob, and raising it makes the task harder
- [ ] Execution feedback and profiler feedback raise each round's success rate; iterative refinement is stronger than repeated sampling at the same budget
- [ ] The Agent's intent must align with the interface's implementation semantics, or the metric drifts (construct drift)
- [ ] Mini SWE loop = test state machine (two-sided check) + edit state machine (edit → test → feedback → retry)


## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.

**Exercise 1: success rate of trajectory repair**

Complete trajectory_success, which computes the solve probability of a single trajectory after r rounds of serial repair.

```python
def trajectory_success(p, r, eta):
    """Single trajectory: first-generation correct probability p; after failure each round fixes with probability eta."""
    return ____                          # complete this

assert abs(trajectory_success(0.2, 2, 0.5) - (1 - 0.8 * 0.5 ** 2)) < 1e-9
assert abs(trajectory_success(0.2, 0, 0.5) - 0.2) < 1e-12
print("Trajectory repair model is correct: after a first-generation failure, each round still has probability eta of a fix.")
```

Hint: the probability that the first generation fails is (1-p); each later round continues to fail with probability (1-eta); the probability that all r rounds fail is $(1-p)(1-eta)^r$; the success probability is 1 minus that.

**Exercise 2: the threshold knob of fast_p**

Complete fast_p, which returns the fraction of candidates that are correct and have speedup greater than p.

```python
def fast_p(cands, p):
    """cands: a list of (is_correct, speedup) pairs."""
    correct = np.array([c[0] for c in cands], dtype=bool)
    speedup = np.array([c[1] for c in cands])
    return ____                          # complete this

cands = [(True, 3.0), (True, 0.5), (False, 4.0), (True, 1.2)]
assert abs(fast_p(cands, 0) - 0.75) < 1e-9
assert abs(fast_p(cands, 1) - 0.5) < 1e-9
assert fast_p(cands, 2) <= fast_p(cands, 1)
print("fast_p is correct: the higher the threshold, the fewer 'correct and fast' candidates.")
```

Hint: join the two boolean conditions with elementwise & , then take mean — note it is & and not and.

**Exercise 3: select a candidate by test vote**

Complete vote_select: passes is a (T, C) boolean matrix, T generated tests' pass/fail on C candidates; lengths is each candidate patch's length. Select the candidate that passed the most tests, breaking ties by the shortest patch.

```python
def vote_select(passes, lengths):
    """Return the index of the candidate with the most test passes; on a tie, the shortest patch."""
    votes = passes.sum(axis=0)
    max_votes = ____                          # complete this: highest vote count
    top = np.where(votes == max_votes)[0]
    return top[np.argmin(____)]               # complete this: among the high-vote group, take the shortest patch

passes = np.array([[1, 1, 0],
                   [1, 1, 0],
                   [0, 0, 1]])
lengths = np.array([40, 25, 60])
assert vote_select(passes, lengths) == 1
print("Vote selector is correct: candidate 1 leads with two votes, and its patch is shorter.")
```

Hint: `passes.sum(axis=0)` is the pass count of each column; first take the subset with the largest pass count, then take the minimum by patch length.

## References

- Ehrlich et al., [CodeMonkeys: Scaling Test-Time Compute for Software Engineering](https://arxiv.org/abs/2501.14723), 2025 — this lecture's first main paper: three-stage decomposition plus two back-to-back state machines, SWE-bench Verified 57.4%
- Ouyang et al., [KernelBench: Can LLMs Write Efficient GPU Kernels?](https://arxiv.org/abs/2502.10517), 2025 — the fast_p metric and the generate-compile-execute-profiler feedback loop; R1's Level 2 fast_1 rose from 36% to 72%
- Trivedi & Nord, [On the Need to Align Intent and Implementation in Uncertainty Quantification for Machine Learning](https://arxiv.org/abs/2506.03037), 2025 — construct drift and the diagnostic discipline of "declare the inference chain before making a claim"
- Wei et al., [Improving Parallel Program Performance with LLM Optimizers via Agent-System Interfaces](https://arxiv.org/abs/2410.15625), ICML 2025 — a landing of the Agent-System interface: DSL plus AutoGuide; 10 iterations beat OpenTuner's thousand iterations
- Jimenez et al., [SWE-bench: Can Language Models Resolve Real-World GitHub Issues?](https://arxiv.org/abs/2310.06770), 2023 — a real GitHub-issue evaluation benchmark, the source of SWE-bench Verified
- Brown et al., [Large Language Monkeys: Scaling Inference Compute with Repeated Sampling](https://arxiv.org/abs/2407.21787), 2024 — the source of "coverage grows approximately log-linearly with samples", the direct predecessor of CodeMonkeys
- Yang et al., [SWE-Agent: Agent-Computer Interfaces Enable Automated Software Engineering](https://arxiv.org/abs/2405.15793), 2024 — another SWE framework that emphasizes the agent-computer interface, complementary to section 3
- Xia et al., [Agentless: Demystifying LLM-based Software Engineering Agents](https://arxiv.org/abs/2407.01489), 2024 — localize-repair-verify three stages, a contrast set for CodeMonkeys
- Tillet et al., [Triton: An Intermediate Language and Compiler for Tiled Neural Network Computations](https://openreview.net/forum?id=RR1J7pJwqB), 2019 — the high-level kernel language KernelBench allows, handling tiling and shared memory implicitly
- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), 2022 — the benchmark of humans writing efficient kernels, cited repeatedly by KernelBench
